In [5]:
# imports

import pandas as pd
from pathlib import Path
import os   
import random
import matplotlib.pyplot as plt

import seaborn as sns

In [6]:
# df = pd.read_csv("sample.csv")
df = pd.read_csv("data.csv")


# Combine age-bucket childcare columns into 3 averaged columns.
# Uses the existing `df` in the notebook.

groups = {
    "MC_0to23_Avg": ["MCBto5", "MC6to11", "MC12to17", "MC18to23"],
    "MC_24to47_Avg": ["MC24to29", "MC30to35", "MC36to41", "MC42to47"],
    "MC_48plus_Avg": ["MC48to53", "MC54toSA"],
}

for new_col, cols in groups.items():
    present = [c for c in cols if c in df.columns]
    if not present:
        # create column of NaNs so downstream code can still reference it
        df[new_col] = pd.NA
        print(f"Warning: none of the columns for {new_col} were found. Created {new_col} with NaN values.")
        continue

    # ensure numeric (coerce errors to NaN), then take mean across the available columns
    df[new_col] = df[present].apply(lambda s: pd.to_numeric(s, errors="coerce")).mean(axis=1)

    missing = [c for c in cols if c not in present]
    if missing:
        print(f"Note: for {new_col} the following expected columns were missing and ignored: {missing}")

# Show the new columns
print(df[[c for c in groups.keys() if c in df.columns]].head())

   MC_0to23_Avg  MC_24to47_Avg  MC_48plus_Avg
0        104.95        90.6775          85.92
1        105.11        91.9700          87.59
2        105.28        93.2650          89.26
3        105.45        94.5600          90.93
4        105.61        95.8525          92.60


In [7]:
short_name_long_name = {
    'State_Name': 'State Name',
    'County_Name': 'County Name',
    'StudyYear': 'Study Year',
    'Households': 'Number of Households',
    
    'MCSA': 'Weekly full-time median price charged for Center-Based Care for school-age children',
    'MFCCSA': 'Weekly full-time median price charged for Family Childcare (home-based) for school-age children',


    'UNR_16':'Unemployment 16 & Older',
    'FUNR_16':'Female Unemployment 16 & Older',
    'MUNR_16':'Male Unemployment 16 & Older',

    'TotalPop':'Total Population',
    
    'UNR_20to64':'Unemployment 20-64 Year Olds',
    'PR_F':'Poverty Rate Families',
    'MHI':'Median Household Income',
    'OneRace_W':'Whites',
    'OneRace_B':'Blacks',
    'OneRace_H':'Hispanic',
    'OneRace_A':'Asian',

    'H_Under6_BothWork':'Households with children under 6 with both parents working',
    'H_6to17_BothWork':'Households with children between 6-17 with both parents working',

    'H_Under6_SingleM':'Households with children under 6 with single mom',
    'MCInfant':'Full Time Median Price for Infant (0 to 23 months) Care',
    'MCToddler':'Full Time Median Price for Toddler (24 to 35 months) Care',
    'MCPreschool':'Full Time Median Price for PreSchool (36 to 54 months) Care',
    
    
    'MME':'Male Median Earnings',
    'FME':'Female Median Earnings',
    
    # By Page
    'MC_0to23_Avg':'Median Childcare Costs for Children 0-23 months',
    'MC_24to47_Avg':'Median Childcare Costs for Children 24-47 months',
    'MC_48plus_Avg':'Median Childcare Costs for Children 48-54 months',
}

In [8]:
# keep only columns that are keys in `short_name_long_name` and present in `df`,
# then rename them to their long names and save to CSV.

present_keys = [k for k in short_name_long_name.keys() if k in df.columns]
if not present_keys:
    raise RuntimeError("No matching columns found in df for keys in short_name_long_name.")

df_filtered = df.loc[:, present_keys].rename(columns=short_name_long_name)
out_path = Path("filtered_renamed.csv")




cols = [
    'Households with children under 6 with both parents working',
    'Households with children between 6-17 with both parents working',
    'Households with children under 6 with single mom'
]
# divide by total households and multiply by 100
for col in cols:
    df_filtered[col] = (df_filtered[col] / df_filtered['Number of Households']) * 100
    df_filtered[col] = df_filtered[col].round(2)







df_filtered.to_csv(out_path, index=False)
print(f"Saved {len(df_filtered.columns)} columns and {len(df_filtered)} rows to: {out_path}")
df_filtered.head()

Saved 28 columns and 34567 rows to: filtered_renamed.csv


,State Name,County Name,Study Year,Number of Households,Weekly full-time median price charged for Center-Based Care for school-age children,Weekly full-time median price charged for Family Childcare (home-based) for school-age children,Unemployment 16 & Older,Female Unemployment 16 & Older,Male Unemployment 16 & Older,Total Population,...,Households with children between 6-17 with both parents working,Households with children under 6 with single mom,Full Time Median Price for Infant (0 to 23 months) Care,Full Time Median Price for Toddler (24 to 35 months) Care,Full Time Median Price for PreSchool (36 to 54 months) Care,Male Median Earnings,Female Median Earnings,Median Childcare Costs for Children 0-23 months,Median Childcare Costs for Children 24-47 months,Median Childcare Costs for Children 48-54 months
0,Alabama,Autauga County,2008,18373,80.92,81.40,5.42,4.41,6.32,49744,...,26.67,5.42,104.95,104.95,85.92,36032.0,21875.0,104.95,90.6775,85.92
1,Alabama,Autauga County,2009,18288,83.42,85.68,5.93,5.72,6.11,49584,...,27.49,6.01,105.11,105.11,87.59,37492.0,22951.0,105.11,91.9700,87.59
2,Alabama,Autauga County,2010,19718,85.92,89.96,6.21,5.57,6.78,53155,...,27.75,5.63,105.28,105.28,89.26,39789.0,23613.0,105.28,93.2650,89.26
3,Alabama,Autauga County,2011,19998,88.43,94.25,7.55,8.13,7.03,53944,...,25.33,5.15,105.45,105.45,90.93,40476.0,23864.0,105.45,94.5600,90.93
4,Alabama,Autauga County,2012,19934,90.93,98.53,8.60,8.88,8.29,54590,...,23.12,5.49,105.61,105.61,92.60,40251.0,25425.0,105.61,95.8525,92.60
